# RNA Velocity: Spliced vs Unspliced Counts

singlify separately quantifies spliced and unspliced (intronic) reads per cell,
enabling RNA velocity analysis without re-processing. These matrices are the
foundation for tools like scVelo and velocyto.

**Outputs**:
- `spliced.1pz` — exonic reads (mature mRNA)
- `unspliced.1pz` — intronic reads (pre-mRNA)
- `exon_counts.1pz` — exon-only counts
- `intron_counts.1pz` — intron-only counts

**Sample**: GSM3573650 (10x v3 PBMC, 75K cells)

In [1]:
import singlet
import numpy as np
from pathlib import Path

SAMPLE = '/mnt/projects/debruinz_project/singlify_pipeline/quant/scrna/GSE125/GSE125416/GSM3573650'

# Load spliced and unspliced matrices
spliced = singlet.read_1pz(str(Path(SAMPLE) / 'spliced.1pz'))
unspliced = singlet.read_1pz(str(Path(SAMPLE) / 'unspliced.1pz'))

print(f'Spliced:   {spliced.n_obs:,} cells × {spliced.n_vars:,} genes')
print(f'Unspliced: {unspliced.n_obs:,} cells × {unspliced.n_vars:,} genes')
print(f'\nTotal spliced UMIs:   {spliced.X.sum():,.0f}')
print(f'Total unspliced UMIs: {unspliced.X.sum():,.0f}')
print(f'Unspliced fraction:   {unspliced.X.sum() / (spliced.X.sum() + unspliced.X.sum()):.1%}')

Spliced:   75,420 cells × 38,606 genes
Unspliced: 75,420 cells × 38,606 genes

Total spliced UMIs:   74,523,962
Total unspliced UMIs: 9,509,015
Unspliced fraction:   11.3%


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Per-cell spliced vs unspliced
spliced_per_cell = np.array(spliced.X.sum(axis=1)).flatten()
unspliced_per_cell = np.array(unspliced.X.sum(axis=1)).flatten()
ratio = unspliced_per_cell / (spliced_per_cell + unspliced_per_cell + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Scatter: spliced vs unspliced
sub = np.random.choice(len(spliced_per_cell), min(5000, len(spliced_per_cell)), replace=False)
axes[0].scatter(spliced_per_cell[sub], unspliced_per_cell[sub], alpha=0.2, s=3, c='#3b82f6')
axes[0].set_xlabel('Spliced UMIs')
axes[0].set_ylabel('Unspliced UMIs')
axes[0].set_title('Spliced vs Unspliced per Cell')

# Histogram of unspliced ratio
axes[1].hist(ratio[ratio > 0], bins=50, color='#22c55e', edgecolor='white')
axes[1].axvline(np.median(ratio[ratio > 0]), color='red', linestyle='--', label=f'Median={np.median(ratio[ratio > 0]):.2f}')
axes[1].set_xlabel('Unspliced Fraction')
axes[1].set_ylabel('Cells')
axes[1].set_title('Unspliced Fraction Distribution')
axes[1].legend()

# Top genes by unspliced counts
unspliced_per_gene = np.array(unspliced.X.sum(axis=0)).flatten()
top_idx = np.argsort(unspliced_per_gene)[::-1][:15]
if hasattr(unspliced, 'var_names') and len(unspliced.var_names) > 0:
    top_genes = [unspliced.var_names[i] for i in top_idx]
else:
    top_genes = [str(i) for i in top_idx]
axes[2].barh(top_genes[::-1], unspliced_per_gene[top_idx][::-1], color='#f59e0b')
axes[2].set_xlabel('Unspliced UMIs')
axes[2].set_title('Top Genes by Unspliced Counts')

plt.suptitle('RNA Velocity Inputs — GSM3573650', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('velocity.png', dpi=150, bbox_inches='tight')
plt.show()

In [3]:
# Summary statistics
print('═' * 50)
print('RNA VELOCITY READINESS')
print('═' * 50)
print(f'  Cells with spliced data:   {(spliced_per_cell > 0).sum():,}')
print(f'  Cells with unspliced data: {(unspliced_per_cell > 0).sum():,}')
print(f'  Median spliced/cell:       {np.median(spliced_per_cell):,.0f}')
print(f'  Median unspliced/cell:     {np.median(unspliced_per_cell):,.0f}')
print(f'  Genes with unspliced:      {(unspliced_per_gene > 0).sum():,} / {unspliced.n_vars:,}')
print(f'  Intronic fraction:         {unspliced.X.sum() / (spliced.X.sum() + unspliced.X.sum()):.1%}')
print('═' * 50)
print('\n→ Ready for scVelo / velocyto analysis!')
print('  Use: spliced.1pz as adata.layers["spliced"]')
print('       unspliced.1pz as adata.layers["unspliced"]')

══════════════════════════════════════════════════
RNA VELOCITY READINESS
══════════════════════════════════════════════════
  Cells with spliced data:   75,420
  Cells with unspliced data: 75,407
  Median spliced/cell:       227
  Median unspliced/cell:     12
  Genes with unspliced:      24,572 / 38,606
  Intronic fraction:         11.3%
══════════════════════════════════════════════════

→ Ready for scVelo / velocyto analysis!
  Use: spliced.1pz as adata.layers["spliced"]
       unspliced.1pz as adata.layers["unspliced"]


## Using with scVelo

```python
import scvelo as scv
import singlet

# Load all layers
adata = singlet.load_dir('/path/to/sample')
adata.layers['spliced'] = singlet.read_1pz('/path/to/sample/spliced.1pz').X
adata.layers['unspliced'] = singlet.read_1pz('/path/to/sample/unspliced.1pz').X

# Standard scVelo pipeline
scv.pp.filter_and_normalize(adata)
scv.pp.moments(adata)
scv.tl.velocity(adata)
scv.tl.velocity_graph(adata)
scv.pl.velocity_embedding(adata, basis='umap')
```

singlify produces velocity-ready matrices for every sample — no additional alignment needed.